# Phase 4: Model Evaluation & Explainability

Runs evaluation for both Model A (risk) and Model B (claims):

Imports necessary packages

In [1]:
import pickle, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, balanced_accuracy_score, recall_score, f1_score,
    precision_score
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130})

MODEL_DIR = Path("/models")
PLOT_DIR  = Path("/phase4_plots")
PLOT_DIR.mkdir(exist_ok=True)

Helpers

In [2]:
def load_model(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def business_metrics(y_true, y_pred, pos_class, model_name, class_label):
    """Print and return recall/precision/F1 for a specific high-stakes class."""
    labels = sorted(y_true.unique())
    idx    = labels.index(pos_class)
    report = classification_report(y_true, y_pred, labels=labels,
                                   target_names=labels, output_dict=True)
    metrics = report[pos_class]
    print(f"\n  ── Business Metric | {model_name} → '{class_label}' class ──")
    print(f"     Recall    : {metrics['recall']:.4f}  "
          f"(of all true {pos_class}, % correctly identified)")
    print(f"     Precision : {metrics['precision']:.4f}")
    print(f"     F1-score  : {metrics['f1-score']:.4f}")
    print(f"     Support   : {int(metrics['support'])}")
    return metrics

def plot_confusion_matrices(model, X_train, y_train, X_test, y_test,
                             labels, title_prefix, filename):
    """Side-by-side train + test confusion matrices."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, X, y, split in zip(axes, [X_train, X_test], [y_train, y_test], ["Train", "Test"]):
        y_pred = model.predict(X)
        cm = confusion_matrix(y, y_pred, labels=labels)
        disp = ConfusionMatrixDisplay(cm, display_labels=labels)
        disp.plot(ax=ax, colorbar=False,
                  cmap="Blues" if split == "Train" else "Greens")
        acc  = accuracy_score(y, y_pred)
        bacc = balanced_accuracy_score(y, y_pred)
        ax.set_title(f"{title_prefix} — {split}\nAcc: {acc:.3f}  |  Balanced Acc: {bacc:.3f}",
                     fontsize=11)
    plt.suptitle(f"{title_prefix} | Confusion Matrices", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / filename, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")

def fairness_segment(model, df_segment, feature_cols, target_col,
                     segment_col, segment_vals, pos_class, model_name):
    """Compute recall for a high-stakes class across demographic segments."""
    results = []
    for val in segment_vals:
        sub = df_segment[df_segment[segment_col] == val].copy()
        if len(sub) < 20:
            continue
        X_sub = sub[feature_cols]
        y_sub = sub[target_col]
        try:
            y_pred = model.predict(X_sub)
            rec = recall_score(y_sub, y_pred, labels=[pos_class],
                               average=None, zero_division=0)
            f1  = f1_score(y_sub, y_pred, labels=[pos_class],
                           average=None, zero_division=0)
            acc = accuracy_score(y_sub, y_pred)
            results.append({
                segment_col:    val,
                "n_records":    len(sub),
                f"recall_{pos_class}":    round(float(rec[0]), 4) if len(rec) else None,
                f"f1_{pos_class}":        round(float(f1[0]),  4) if len(f1)  else None,
                "accuracy":     round(acc, 4),
            })
        except Exception as e:
            results.append({segment_col: val, "error": str(e)})
    return pd.DataFrame(results)

Loading the data & rebuilding splits (mirrors model scripts)

In [3]:
print("\n" + "="*60)
print("  PHASE 4 | Model Evaluation & Explainability")
print("="*60)

df = pd.read_parquet("merged_clean.parquet")
df["visit_date"]    = pd.to_datetime(df["visit_date"],    errors="coerce")
df["billing_date"]  = pd.to_datetime(df["billing_date"],  errors="coerce")
df["registration_date"] = pd.to_datetime(df["registration_date"], errors="coerce")


  PHASE 4 | Model Evaluation & Explainability


Re-engineer features (in case of fresh session)

In [4]:
df["visit_frequency"]         = df.groupby("patient_id")["visit_id"].transform("count")
df["avg_los_per_patient"]     = df.groupby("patient_id")["length_of_stay_hours"].transform("mean").round(2)
df["days_since_registration"] = (df["visit_date"] - df["registration_date"]).dt.days
prov_rej = (df.groupby("insurance_provider")["claim_status"]
            .apply(lambda s: round((s=="Rejected").sum()/len(s)*100, 2))
            .reset_index(name="provider_rejection_rate"))
df = df.merge(prov_rej, on="insurance_provider", how="left")
df["visit_month"]     = df["visit_date"].dt.month
df["visit_quarter"]   = df["visit_date"].dt.quarter
df["visit_dayofweek"] = df["visit_date"].dt.dayofweek
df["is_weekend"]      = (df["visit_dayofweek"] >= 5).astype(int)
df["high_risk_flag"]  = (df["risk_score"] == "High").astype(int)
df["billed_per_hour"] = (df["billed_amount"] / df["length_of_stay_hours"].replace(0, np.nan)).round(2)
df["above_avg_bill"]  = (df["billed_amount"] > df["billed_amount"].mean()).astype(int)

print(f"  Dataset loaded: {df.shape}")

  Dataset loaded: (24961, 32)


# Model A feature schema

In [5]:
A_NUMERIC = ["age","chronic_flag","length_of_stay_hours","visit_frequency",
             "avg_los_per_patient","days_since_registration","provider_rejection_rate",
             "visit_month","visit_quarter","visit_dayofweek","is_weekend"]
A_CAT     = ["department","visit_type","gender","city","insurance_provider"]
A_FEAT    = A_NUMERIC + A_CAT
A_TARGET  = "risk_score"

# Model B feature schema

In [6]:
B_NUMERIC = ["age","chronic_flag","length_of_stay_hours","billed_amount",
             "billed_per_hour","above_avg_bill","visit_frequency",
             "avg_los_per_patient","days_since_registration","provider_rejection_rate",
             "visit_month","visit_quarter","visit_dayofweek","is_weekend","high_risk_flag"]
B_CAT     = ["department","visit_type","gender","city","insurance_provider"]
B_FEAT    = B_NUMERIC + B_CAT
B_TARGET  = "claim_status"

Rebuilding Model A time split

In [9]:
df["visit_frequency"]         = df.groupby("patient_id")["visit_id"].transform("count")
df["avg_los_per_patient"]     = df.groupby("patient_id")["length_of_stay_hours"].transform("mean").round(2)
df["days_since_registration"] = (df["visit_date"] - df["registration_date"]).dt.days
prov_rej = (df.groupby("insurance_provider")["claim_status"]
            .apply(lambda s: round((s=="Rejected").sum()/len(s)*100, 2))
            .reset_index(name="provider_rejection_rate"))
df = df.merge(prov_rej, on="insurance_provider", how="left")
df["visit_month"]     = df["visit_date"].dt.month
df["visit_quarter"]   = df["visit_date"].dt.quarter
df["visit_dayofweek"] = df["visit_date"].dt.dayofweek
df["is_weekend"]      = (df["visit_dayofweek"] >= 5).astype(int)
df["high_risk_flag"]  = (df["risk_score"] == "High").astype(int)
df["billed_per_hour"] = (df["billed_amount"] / df["length_of_stay_hours"].replace(0, np.nan)).round(2)
df["above_avg_bill"]  = (df["billed_amount"] > df["billed_amount"].mean()).astype(int)

a_df = df[A_FEAT + [A_TARGET, "visit_date"]].dropna(subset=[A_TARGET,"visit_date"])
a_df = a_df.dropna(subset=A_NUMERIC).sort_values("visit_date").reset_index(drop=True)
a_split = int(len(a_df) * 0.80)
a_train = a_df.iloc[:a_split];  a_test = a_df.iloc[a_split:]
Xa_train = a_train[A_FEAT];     Xa_test = a_test[A_FEAT]
ya_train = a_train[A_TARGET];   ya_test = a_test[A_TARGET]

Rebuilding Model B time split

In [10]:
df["split_date"] = df["billing_date"].fillna(df["visit_date"])
b_df = df[B_FEAT + [B_TARGET, "split_date"]].dropna(subset=[B_TARGET,"split_date"])
b_df = b_df.dropna(subset=B_NUMERIC).sort_values("split_date").reset_index(drop=True)
b_split = int(len(b_df) * 0.80)
b_train = b_df.iloc[:b_split];  b_test = b_df.iloc[b_split:]
Xb_train = b_train[B_FEAT];     Xb_test = b_test[B_FEAT]
yb_train = b_train[B_TARGET];   yb_test = b_test[B_TARGET]

print(f"  Model A — train:{len(a_train):,}  test:{len(a_test):,}")
print(f"  Model B — train:{len(b_train):,}  test:{len(b_test):,}")

  Model A — train:19,968  test:4,993
  Model B — train:19,968  test:4,993


Loading the best models

In [11]:
model_a = load_model(MODEL_DIR / "model_a_tuned.pkl")
model_b = load_model(MODEL_DIR / "model_b_tuned.pkl")
print("  Models loaded.")

FileNotFoundError: [Errno 2] No such file or directory: '\\models\\model_a_tuned.pkl'

# Section-1 Model A: Confusion matrices + Classification report

In [ ]:
print("\n" + "─"*60)
print("  SECTION 1 | Model A — Risk Classification Evaluation")
print("─"*60)

a_labels = sorted(ya_test.unique())
plot_confusion_matrices(model_a, Xa_train, ya_train, Xa_test, ya_test,
                        a_labels, "Model A (Risk)", "p4_model_a_cm.png")

ya_pred_test  = model_a.predict(Xa_test)
ya_pred_train = model_a.predict(Xa_train)

print("\n  ── Test Set Classification Report ──")
print(classification_report(ya_test, ya_pred_test))

print(f"  Train Accuracy : {accuracy_score(ya_train, ya_pred_train):.4f}")
print(f"  Test  Accuracy : {accuracy_score(ya_test,  ya_pred_test):.4f}")
print(f"  Train Balanced : {balanced_accuracy_score(ya_train, ya_pred_train):.4f}")
print(f"  Test  Balanced : {balanced_accuracy_score(ya_test,  ya_pred_test):.4f}")

# Business metric — High Risk recall is the critical KPI
bm_a = business_metrics(ya_test, ya_pred_test, "High", "Model A", "High Risk")

# Overfitting check
gap = accuracy_score(ya_train, ya_pred_train) - accuracy_score(ya_test, ya_pred_test)
print(f"\n  Train-Test Accuracy Gap : {gap:.4f}  "
      f"({'⚠️ Possible overfit' if gap > 0.05 else '✅ Acceptable'})")

# Section-2 Model B: Confusion matrices + Classification report

In [ ]:
print("\n" + "─"*60)
print("  SECTION 2 | Model B — Claim Outcome Evaluation")
print("─"*60)

b_labels = sorted(yb_test.unique())
plot_confusion_matrices(model_b, Xb_train, yb_train, Xb_test, yb_test,
                        b_labels, "Model B (Claims)", "p4_model_b_cm.png")

yb_pred_test  = model_b.predict(Xb_test)
yb_pred_train = model_b.predict(Xb_train)

print("\n  ── Test Set Classification Report ──")
print(classification_report(yb_test, yb_pred_test))

print(f"  Train Accuracy : {accuracy_score(yb_train, yb_pred_train):.4f}")
print(f"  Test  Accuracy : {accuracy_score(yb_test,  yb_pred_test):.4f}")
print(f"  Train Balanced : {balanced_accuracy_score(yb_train, yb_pred_train):.4f}")
print(f"  Test  Balanced : {balanced_accuracy_score(yb_test,  yb_pred_test):.4f}")

# Business metric — Rejected recall is the critical KPI
bm_b = business_metrics(yb_test, yb_pred_test, "Rejected", "Model B", "Rejected Claim")

gap_b = accuracy_score(yb_train, yb_pred_train) - accuracy_score(yb_test, yb_pred_test)
print(f"\n  Train-Test Accuracy Gap : {gap_b:.4f}  "
      f"({'⚠️ Possible overfit' if gap_b > 0.05 else '✅ Acceptable'})")

# Section-3: Feature Importance Plots

In [ ]:
print("\n" + "─"*60)
print("  SECTION 3 | Feature Importance")
print("─"*60)

def plot_importance(model, numeric_feats, cat_feats, clf_attr,
                    title, filename, color):
    pre  = model.named_steps["preprocessor"]
    clf  = model.named_steps[clf_attr]
    ohe  = list(pre.named_transformers_["cat"].get_feature_names_out(cat_feats))
    names = numeric_feats + ohe
    imp  = pd.Series(clf.feature_importances_, index=names)
    top20 = imp.sort_values(ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    top20.sort_values().plot(kind="barh", ax=ax, color=color, edgecolor="white")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Importance Score")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / filename, bbox_inches="tight")
    plt.show()
    print(f"  Top 5 features: {list(top20.head(5).index)}")
    print(f"  Saved: {filename}")
    return top20

print("\n  Model A — Gradient Boosting Feature Importance:")
top_a = plot_importance(model_a, A_NUMERIC, A_CAT, "classifier",
                        "Model A — Top 20 Features (Risk Classification)",
                        "p4_model_a_importance.png", "steelblue")

print("\n  Model B — Random Forest Feature Importance:")
top_b = plot_importance(model_b, B_NUMERIC, B_CAT, "classifier",
                        "Model B — Top 20 Features (Claim Outcome)",
                        "p4_model_b_importance.png", "seagreen")

# Section-4: Fairness Segmentation

In [ ]:
print("\n" + "─"*60)
print("  SECTION 4 | Fairness Segmentation")
print("─"*60)

# Attach predictions to test sets
a_test_eval = a_test.copy()
b_test_eval = b_test.copy()

# ── Model A fairness ─────────────────────────────────────────────────────────
print("\n  Model A — High Risk Recall by Segment:\n")

for seg_col in ["gender", "insurance_provider", "city"]:
    vals = a_test_eval[seg_col].dropna().unique()
    seg_df = fairness_segment(
        model_a, a_test_eval, A_FEAT, A_TARGET,
        seg_col, vals, "High", "Model A"
    )
    seg_df = seg_df.sort_values("recall_High", ascending=False)
    print(f"  By {seg_col}:")
    print(seg_df.to_string(index=False))

    # Plot
    if seg_col != "city" or len(vals) <= 20:
        fig, ax = plt.subplots(figsize=(10, max(4, len(vals)*0.4)))
        seg_df_plot = seg_df.dropna(subset=["recall_High"]).sort_values("recall_High")
        ax.barh(seg_df_plot[seg_col].astype(str),
                seg_df_plot["recall_High"], color="steelblue", edgecolor="white")
        ax.axvline(seg_df_plot["recall_High"].mean(), color="red",
                   linestyle="--", linewidth=1.5, label="Mean recall")
        ax.set_title(f"Model A — High Risk Recall by {seg_col}", fontsize=11)
        ax.set_xlabel("Recall (High Risk)")
        ax.legend()
        plt.tight_layout()
        fname = f"p4_modelA_fairness_{seg_col}.png"
        plt.savefig(PLOT_DIR / fname, bbox_inches="tight")
        plt.show()
        print(f"  Saved: {fname}\n")

# ── Model B fairness ─────────────────────────────────────────────────────────
print("\n  Model B — Rejected Recall by Segment:\n")

for seg_col in ["gender", "insurance_provider", "city"]:
    vals = b_test_eval[seg_col].dropna().unique()
    seg_df = fairness_segment(
        model_b, b_test_eval, B_FEAT, B_TARGET,
        seg_col, vals, "Rejected", "Model B"
    )
    seg_df = seg_df.sort_values("recall_Rejected", ascending=False)
    print(f"  By {seg_col}:")
    print(seg_df.to_string(index=False))

    if seg_col != "city" or len(vals) <= 20:
        fig, ax = plt.subplots(figsize=(10, max(4, len(vals)*0.4)))
        seg_df_plot = seg_df.dropna(subset=["recall_Rejected"]).sort_values("recall_Rejected")
        ax.barh(seg_df_plot[seg_col].astype(str),
                seg_df_plot["recall_Rejected"], color="seagreen", edgecolor="white")
        ax.axvline(seg_df_plot["recall_Rejected"].mean(), color="red",
                   linestyle="--", linewidth=1.5, label="Mean recall")
        ax.set_title(f"Model B — Rejected Recall by {seg_col}", fontsize=11)
        ax.set_xlabel("Recall (Rejected)")
        ax.legend()
        plt.tight_layout()
        fname = f"p4_modelB_fairness_{seg_col}.png"
        plt.savefig(PLOT_DIR / fname, bbox_inches="tight")
        plt.show()
        print(f"  Saved: {fname}\n")

# Section-5: Explainability Summary

In [ ]:
print("\n" + "─"*60)
print("  SECTION 5 | SHAP Explainability")
print("─"*60)

try:
    import shap

    def shap_summary(model, X_sample, numeric_feats, cat_feats,
                     clf_attr, title, filename):
        pre   = model.named_steps["preprocessor"]
        clf   = model.named_steps[clf_attr]
        X_t   = pre.transform(X_sample)
        ohe   = list(pre.named_transformers_["cat"].get_feature_names_out(cat_feats))
        names = numeric_feats + ohe

        explainer = shap.TreeExplainer(clf)
        shap_vals = explainer.shap_values(X_t)

        # For multiclass, shap_vals is a list or 3D array; plot mean abs across classes
        if isinstance(shap_vals, list):
            # If it's a list of 2D arrays (samples, features), average across the list elements (classes)
            mean_abs = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
        else:
            # If it's a 3D array (samples, features, classes), average along the last axis (classes)
            mean_abs = np.mean(np.abs(shap_vals), axis=2)

        shap_df = pd.DataFrame(mean_abs, columns=names)
        top_shap = shap_df.mean().sort_values(ascending=False).head(15)

        fig, ax = plt.subplots(figsize=(10, 6))
        top_shap.sort_values().plot(kind="barh", ax=ax,
                                    color="darkorange", edgecolor="white")
        ax.set_title(f"SHAP — Mean |SHAP value| — {title}", fontsize=11)
        ax.set_xlabel("Mean |SHAP value|")
        plt.tight_layout()
        plt.savefig(PLOT_DIR / filename, bbox_inches="tight")
        plt.show()
        print(f"  Top SHAP features: {list(top_shap.head(5).index)}")
        print(f"  Saved: {filename}")

    sample_a = Xa_test.sample(min(500, len(Xa_test)), random_state=42)
    sample_b = Xb_test.sample(min(500, len(Xb_test)), random_state=42)

    print("\n  Model A SHAP:")
    try:
        shap_summary(model_a, sample_a, A_NUMERIC, A_CAT,
                     "classifier", "Model A Risk", "p4_shap_model_a.png")
    except Exception as e:
        # Catching generic Exception for robustness, as specific InvalidModelError might be nested
        if "GradientBoostingClassifier is only supported for binary classification" in str(e):
            print(f"  Skipping SHAP for Model A: {e}")
            print("  shap.TreeExplainer for GradientBoostingClassifier does not currently support multiclass models.")
            print("  Consider using shap.KernelExplainer or training binary models for SHAP analysis if multiclass support is critical.")
        else:
            raise # Re-raise if it's an unexpected error


    print("\n  Model B SHAP:")
    shap_summary(model_b, sample_b, B_NUMERIC, B_CAT,
                 "classifier", "Model B Claims", "p4_shap_model_b.png")

except ImportError:
    print("  SHAP not installed. Run: pip install shap")
    print("  Falling back to built-in feature importance (already plotted in Section 3).")

# Section-6: Saving evaluation metrics for Report

In [ ]:
metrics_out = {
    "model_a": {
        "train_accuracy":          round(accuracy_score(ya_train, ya_pred_train), 4),
        "test_accuracy":           round(accuracy_score(ya_test,  ya_pred_test),  4),
        "train_balanced_accuracy": round(balanced_accuracy_score(ya_train, ya_pred_train), 4),
        "test_balanced_accuracy":  round(balanced_accuracy_score(ya_test,  ya_pred_test),  4),
        "high_risk_recall":        round(bm_a["recall"],    4),
        "high_risk_precision":     round(bm_a["precision"], 4),
        "high_risk_f1":            round(bm_a["f1-score"],  4),
        "train_test_gap":          round(gap, 4),
    },
    "model_b": {
        "train_accuracy":          round(accuracy_score(yb_train, yb_pred_train), 4),
        "test_accuracy":           round(accuracy_score(yb_test,  yb_pred_test),  4),
        "train_balanced_accuracy": round(balanced_accuracy_score(yb_train, yb_pred_train), 4),
        "test_balanced_accuracy":  round(balanced_accuracy_score(yb_test,  yb_pred_test),  4),
        "rejected_recall":         round(bm_b["recall"],    4),
        "rejected_precision":      round(bm_b["precision"], 4),
        "rejected_f1":             round(bm_b["f1-score"],  4),
        "train_test_gap":          round(gap_b, 4),
    },
}

with open(MODEL_DIR / "phase4_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
print("\n  Saved: phase4_metrics.json")

print("\n" + "="*60)
print("  Phase 4 evaluation complete.")
print(f"  All plots saved to: {PLOT_DIR}")
print("="*60 + "\n")